In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DecimalType,
    DateType,
    TimestampType,
    BooleanType
)

In [2]:
spark = SparkSession.builder.appName("EDA").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/06 21:35:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
date_key VARCHAR(10), year INT, month INT, week INT,
day_of_week VARCHAR(10), is_weekend VARCHAR(5),
is_holiday VARCHAR(5), holiday_name VARCHAR(100) NULL

In [ ]:
order_items_schema = StructType([
    StructField("APP_NAME",StringType()),
    StructField("RESTAURANT_ID",StringType()),
    StructField("CREATION_TIME_UTC",TimestampType()),
    StructField("ORDER_ID",StringType()),
    StructField("USER_ID",StringType()),
    StructField("PRINTED_CARD_NUMBER",StringType()),
    StructField("IS_LOYALTY",StringType()),
    StructField("CURRENCY",StringType()),
    StructField("LINEITEM_ID",StringType()),
    StructField("ITEM_CATEGORY",StringType()),
    StructField("ITEM_NAME",StringType()),
    StructField("ITEM_PRICE",DecimalType(10,2)),
    StructField("ITEM_QUANTITY",IntegerType())
    
])



order_item_options_schema = StructType([
    StructField("ORDER_ID",StringType()),
    StructField("LINEITEM_ID",StringType()),
    StructField("OPTION_GROUP_NAME",StringType()),
    StructField("OPTION_NAME",StringType()),
    StructField("OPTION_PRICE",DecimalType(10,2)),
    StructField("OPTION_QUANTITY",IntegerType())
    
])


date_dim_schema = StructType([
    StructField("date_key",StringType()),
    StructField("year",IntegerType()),
    StructField("month",IntegerType()),
    StructField("week",IntegerType()),
    StructField("day_of_week",StringType()),
    StructField("is_holiday",StringType()),
    StructField("is_weekend",StringType()),
    StructField("holiday_name",StringType()),
    
])

In [ ]:
/Users/saim/Downloads/DE PROJECTS/Business Insights Assessment/order_items.csv

In [ ]:
from pyspark.sql import functions as F
from functools import reduce

def quarantine_corrupt_order_items(df,required_columns,qurantine_path):

    df.sparkSession.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    condition = reduce(lambda x, y: x | y,
                       [F.col(c).isNull() for c in required_columns]
                       )
    corrupt_df = df.filter(condition)

    reject_reason = F.concat_ws(", ",*[F.when(F.col(c).isNull(),F.lit(f"{c} is NULL"))
                                       for c in required_columns]
                                )
    corrupt_df = corrupt_df.withColumn("REJECT_REASON",reject_reason)

    valid_order_df = df.filter(~condition)



    corrupt_df.write \
        .mode("overwrite") \
        .partitionBy("load_date") \
        .format("parquet") \
        .save(qurantine_path)
         

    return valid_order_df,corrupt_df

In [ ]:
reduce(lambda x,y: x & y,
       [
           F.col(f"{left_alias}.{key}") == F.col(f"{right_alias}.{key}")
           for c in join_keys

       ] 
       )

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce


def build_join_condition(left_alias: str, right_alias: str, join_keys: list[str]):
    return reduce(
        lambda a, b: a & b,
        [
            F.col(f"{left_alias}.{key}") == F.col(f"{right_alias}.{key}")
            for key in join_keys
        ]
    )


def quarantine_orphan_options(
    options_df: DataFrame,
    valid_orders_df: DataFrame,
    join_keys: list[str],
    quarantine_path: str,
    load_date: str
) -> tuple[DataFrame, DataFrame]:
    """
    Finds option rows that do not have a matching valid order.

    Returns:
        valid_options_df: options with matching orders
        orphan_options_df: options without matching orders
    """
    options_df.sparkSession.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    orders = valid_orders_df.alias("orders")
    options = options_df.alias("options")

    join_condition = build_join_condition(
        left_alias="options",
        right_alias="orders",
        join_keys=join_keys
    )

    orphan_options_df = (
        options
        .join(orders, on=join_condition, how="left_anti")
        .withColumn("reject_reason", F.lit("ORPHAN_OPTION_NO_MATCHING_ORDER"))
        .withColumn("load_date", F.lit(load_date))
    )

    valid_options_df = (
        options
        .join(orders, on=join_condition, how="left_semi")
    )

    (
        orphan_options_df
        .write
        .mode("overwrite")
        .partitionBy("load_date")
        .format("parquet")
        .save(quarantine_path)
    )

    return valid_options_df, orphan_options_df

In [ ]:
def preaggregate_options(valid_options_df):
    options_grouped = valid_options_df.groupBy("ORDER_ID","LINEITEM_ID") \
                                      .agg(
                                            (F.sum(F.col("OPTION_PRICE") * F.col("OPTION_QUANTITY"))
                                            .alias("TOTAL_OPTIONS_PRICE"))
                                            ,F.count("*").alias("NUM_OPTIONS")
                                            )
    return options_grouped
    


def compute_line_revenue(valid_items_df, options_agg_df, join_keys):


    orders = valid_items_df.alias("orders")
    options = options_agg_df.alias("options")

    join_condition = build_join_condition(
        left_alias="options",
        right_alias="orders",
        join_keys=join_keys
    )


    joined_df = orders.join(options,
                                    on = join_condition
                                    ,how ='left') \
                                    .drop(options['ORDER_ID']) \
                                    .drop(options['LINEITEM_ID'])
    
    return joined_df



def compute_line_revenue(valid_items_df, options_agg_df, join_keys):
    joined_df = valid_items_df.join(options_agg_df, on=join_keys, how="left")

    joined_df = joined_df.withColumn(
        "LINE_REVENUE",
        F.col("ITEM_PRICE") * F.col("ITEM_QUANTITY")
        + F.coalesce(F.col("TOTAL_OPTIONS_PRICE"), F.lit(0)),
    )
    return joined_df

In [ ]:
def derive_order_date(df):
    df = df.withColumn("ORDER_DATE",F.to_date("CREATION_TIME_UTC"))
    return df


def write_to_prod(df, prod_path,table_name):

        df.sparkSession.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
        target = f"{prod_path}/{table_name}"
        (
        df
        .write
        .mode("overwrite")
        .partitionBy("ORDER_DATE")
        .format("parquet")
        .save(target)
    )
    
        return target

In [5]:
order_items = spark.read.option("header","true") \
                        .option("inferSchema","true") \
                        .option("multiLine", "true") \
                        .csv("/Users/saim/Downloads/DE PROJECTS/Business Insights Assessment/order_items.csv")


order_item_options = spark.read \
                          .option("header","true") \
                          .option("inferSchema","true") \
                          .option("multiLine","true") \
                          .csv("/Users/saim/Downloads/DE PROJECTS/Business Insights Assessment/order_item_options.csv")


date_dim = spark.read \
                .option("header","true") \
                .option("inferSchema","true") \
                .option("multiLine","true") \
                .csv("/Users/saim/Downloads/DE PROJECTS/Business Insights Assessment/date_dim.csv")

In [7]:
keys = ['ORDER_ID','LINEITEM_ID']

df_corrupt = order_items.filter((F.col("ORDER_ID").isNull()) | (F.col("LINEITEM_ID").isNull()))

In [8]:
df_corrupt.show()

+-------------+--------------------+--------------------+--------------------+--------------------+-------------------+----------+--------+-----------+-------------+---------+----------+-------------+
|     APP_NAME|       RESTAURANT_ID|   CREATION_TIME_UTC|            ORDER_ID|             USER_ID|PRINTED_CARD_NUMBER|IS_LOYALTY|CURRENCY|LINEITEM_ID|ITEM_CATEGORY|ITEM_NAME|ITEM_PRICE|ITEM_QUANTITY|
+-------------+--------------------+--------------------+--------------------+--------------------+-------------------+----------+--------+-----------+-------------+---------+----------+-------------+
|Alltown Fresh|5e7e35ed902ad5ac0...|2021-12-07 10:23:...|61af8a82ffd125453...|609d681462e498b35...|               NULL|     false|     USD|       NULL|         NULL|     NULL|      4.39|            0|
+-------------+--------------------+--------------------+--------------------+--------------------+-------------------+----------+--------+-----------+-------------+---------+----------+----------

In [4]:
date_dim.schema['date_key'].dataType

StringType()

In [5]:
order_items = order_items.withColumn("ORDER_DATE",F.to_date("CREATION_TIME_UTC"))
date_dim = date_dim.withColumn("DATE",F.to_date(F.col("date_key"),"dd-MM-yyyy"))

In [23]:
date_dim.show(20)

+----------+----+-----+----+-----------+----------+----------+--------------------+----------+
|  date_key|year|month|week|day_of_week|is_weekend|is_holiday|        holiday_name|      DATE|
+----------+----+-----+----+-----------+----------+----------+--------------------+----------+
|01-01-2023|2023|    1|  52|     Sunday|      true|      true|      New Year's Day|2023-01-01|
|02-01-2023|2023|    1|   1|     Monday|     false|      true|New Year's Day Ob...|2023-01-02|
|03-01-2023|2023|    1|   1|    Tuesday|     false|     false|                NULL|2023-01-03|
|04-01-2023|2023|    1|   1|  Wednesday|     false|     false|                NULL|2023-01-04|
|05-01-2023|2023|    1|   1|   Thursday|     false|     false|                NULL|2023-01-05|
|06-01-2023|2023|    1|   1|     Friday|     false|     false|                NULL|2023-01-06|
|07-01-2023|2023|    1|   1|   Saturday|      true|     false|                NULL|2023-01-07|
|08-01-2023|2023|    1|   1|     Sunday|      true

In [ ]:
date_dim.select("DATE").count()

365

In [ ]:
order_item_options.show()

+--------------------+--------------------+--------------------+--------------------+------------+---------------+
|            ORDER_ID|         LINEITEM_ID|   OPTION_GROUP_NAME|         OPTION_NAME|OPTION_PRICE|OPTION_QUANTITY|
+--------------------+--------------------+--------------------+--------------------+------------+---------------+
|6278eb58dfafab1e0...|6278eb63dfafab1e0...|              Cheese|     American Cheese|         0.0|              1|
|6278eb58dfafab1e0...|6278eb63dfafab1e0...|             Veggies|             Lettuce|         0.5|              1|
|6278eb58dfafab1e0...|6278eb63dfafab1e0...|             Veggies|            Tomatoes|         0.0|              1|
|6278eb58dfafab1e0...|6278eb63dfafab1e0...|             Veggies|               Onion|         0.0|              1|
|6278eb58dfafab1e0...|6278eb63dfafab1e0...|             Veggies|       Green Peppers|         0.0|              1|
|627b87809f91930b3...|627b90208270026f7...|              Cheese|     American Ch

In [ ]:
order_items_null_counts = (order_items.select([F.sum(F.when(F.col(c).isNull(),1).otherwise(0)).alias(c)
                                   for c in order_items.columns])
                                   .first()
                                   .asDict()
                                   
                                   )
order_item_options_nulls = (

                      order_item_options.select([F.sum(F.when(F.col(c).isNull(),1).otherwise(0)).alias(c)
                                                 for c in order_item_options.columns]).first().asDict()  

)

date_dim_nulls = (
                    date_dim.select([F.sum(F.when(F.col(c).isNull(),1).otherwise(0)).alias(c)
                                     for c in date_dim.columns]).first().asDict()

)

26/07/04 19:30:51 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [ ]:
order_item_options.groupBy("ORDER_ID","LINEITEM_ID").count().filter(F.col('count')>1).show(truncate=False)

+------------------------+------------------------+-----+
|ORDER_ID                |LINEITEM_ID             |count|
+------------------------+------------------------+-----+
|64bc0912a1340dd0920f30cf|64bc09d25f05ac24e7029b12|3    |
|62824646901170338004aaa4|62824656720d3169a8392107|4    |
|64e0b0543cdf02a21100b5fe|64e0b08ef06bbc6286044677|2    |
|61e6feb726ac063b894414c5|61e6fec975333839113be1ba|2    |
|650f017643403163fe040b2c|650f0176cc21574d0500f9cc|2    |
|646a13420b68aa455f0ea438|646a13428fe91f6d5e0a5832|2    |
|64f0b923dc14d35d080febf3|64f0bc48c3622836b2088a57|2    |
|5f7b236aadb0d522157b244a|5f7b2370505ee93f267b23c7|9    |
|61df7ac8534ed9226e231ba3|61df7affb207716e71762396|3    |
|65c4ec133cb1fcc23c04431f|65c4ec3044d2d198b40dda4f|2    |
|62d01ffff0284f1d0b612e40|62d01fff6707f90ee9100294|2    |
|629a4b7c9d20aad562035d4a|629a4b7daa0ae63eb0052927|4    |
|6481f8da8fb4783fc208a6d2|6481f8da81080942f5079110|2    |
|62aa65f60d96c66d8e7e53c2|62aa65f6bcb2eecdfe05635b|2    |
|64d60f29b94be

In [ ]:
filtered_items = order_item_options.filter((F.col("ORDER_ID") == '61e6feb726ac063b894414c5')
                          & (F.col("LINEITEM_ID") == '61e6fec875333839113be1b8'))
filtered_items.show(truncate=False)

+------------------------+------------------------+----------------------------+------------------+------------+---------------+
|ORDER_ID                |LINEITEM_ID             |OPTION_GROUP_NAME           |OPTION_NAME       |OPTION_PRICE|OPTION_QUANTITY|
+------------------------+------------------------+----------------------------+------------------+------------+---------------+
|61e6feb726ac063b894414c5|61e6fec875333839113be1b8|Cracked Oats Oatmeal Options|Add Fresh Bananas |0.5         |1              |
|61e6feb726ac063b894414c5|61e6fec875333839113be1b8|Cracked Oats Oatmeal Options|Add Chopped Pecans|1.0         |1              |
|61e6feb726ac063b894414c5|61e6fec875333839113be1b8|Cracked Oats Oatmeal Options|Add Dried Fruit   |1.0         |1              |
+------------------------+------------------------+----------------------------+------------------+------------+---------------+



In [ ]:
filtered_items.groupBy("ORDER_ID","LINEITEM_ID").agg(F.sum(F.col("OPTION_PRICE") * F.col("OPTION_QUANTITY")).alias("TOTAL_OPTIONS_PRICE")
                                                     
                                                     ).show()

+--------------------+--------------------+-------------------+
|            ORDER_ID|         LINEITEM_ID|TOTAL_OPTIONS_PRICE|
+--------------------+--------------------+-------------------+
|61e6feb726ac063b8...|61e6fec8753338391...|                2.5|
+--------------------+--------------------+-------------------+



In [ ]:
order_item_options.show(1)

+--------------------+--------------------+-----------------+---------------+------------+---------------+
|            ORDER_ID|         LINEITEM_ID|OPTION_GROUP_NAME|    OPTION_NAME|OPTION_PRICE|OPTION_QUANTITY|
+--------------------+--------------------+-----------------+---------------+------------+---------------+
|6278eb58dfafab1e0...|6278eb63dfafab1e0...|           Cheese|American Cheese|         0.0|              1|
+--------------------+--------------------+-----------------+---------------+------------+---------------+
only showing top 1 row



In [25]:
order_item_options_grouped = order_item_options.groupBy("ORDER_ID","LINEITEM_ID") \
.agg((F.sum(F.col("OPTION_PRICE") * F.col("OPTION_QUANTITY")).alias("TOTAL_OPTIONS_PRICE")),F.count("*").alias("NUM_OPTIONS")
                                            )

In [26]:
order_item_options_grouped.show()

+--------------------+--------------------+-------------------+-----------+
|            ORDER_ID|         LINEITEM_ID|TOTAL_OPTIONS_PRICE|NUM_OPTIONS|
+--------------------+--------------------+-------------------+-----------+
|64bc0912a1340dd09...|64bc09d25f05ac24e...|                0.0|          3|
|62824646901170338...|62824656720d3169a...|                0.0|          4|
|64e0b0543cdf02a21...|64e0b08ef06bbc628...|                3.0|          2|
|61e6feb726ac063b8...|61e6fec9753338391...|                0.0|          2|
|653004bcd443d42e1...|653004e0f406c869c...|                2.0|          1|
|650f017643403163f...|650f0176cc21574d0...|                0.0|          2|
|639727f5bd4e2ce6b...|63972b8029d82708c...|                0.0|          1|
|646a13420b68aa455...|646a13428fe91f6d5...|                4.0|          2|
|64f0b923dc14d35d0...|64f0bc48c3622836b...|                2.0|          2|
|630794e59b5f56056...|630794f3ed375fe95...|                1.0|          1|
|5f7b236aadb

In [ ]:
order_items.filter(F.col("ORDER_ID") == '63d5d88aa26a29fa550df31f').show()

+-------------+--------------------+--------------------+--------------------+--------------------+-------------------+----------+--------+--------------------+-------------+-------------------+----------+-------------+----------+
|     APP_NAME|       RESTAURANT_ID|   CREATION_TIME_UTC|            ORDER_ID|             USER_ID|PRINTED_CARD_NUMBER|IS_LOYALTY|CURRENCY|         LINEITEM_ID|ITEM_CATEGORY|          ITEM_NAME|ITEM_PRICE|ITEM_QUANTITY|ORDER_DATE|
+-------------+--------------------+--------------------+--------------------+--------------------+-------------------+----------+--------+--------------------+-------------+-------------------+----------+-------------+----------+
|Alltown Fresh|5fece84aadb0d5150...|2023-01-28 20:23:...|63d5d88aa26a29fa5...|5ece77fe902ad5013...|               NULL|     false|     USD|63d5d88d7a15b80ac...|    Breakfast|Bacon Egg & Cheese*|      5.99|            1|2023-01-28|
+-------------+--------------------+--------------------+-------------------

In [ ]:
order_items_null_counts

{'APP_NAME': 0,
 'RESTAURANT_ID': 0,
 'CREATION_TIME_UTC': 0,
 'ORDER_ID': 0,
 'USER_ID': 17808,
 'PRINTED_CARD_NUMBER': 157435,
 'IS_LOYALTY': 0,
 'CURRENCY': 0,
 'LINEITEM_ID': 1,
 'ITEM_CATEGORY': 1,
 'ITEM_NAME': 1,
 'ITEM_PRICE': 0,
 'ITEM_QUANTITY': 0}

In [ ]:
order_item_options_nulls

{'ORDER_ID': 0,
 'LINEITEM_ID': 0,
 'OPTION_GROUP_NAME': 0,
 'OPTION_NAME': 0,
 'OPTION_PRICE': 0,
 'OPTION_QUANTITY': 0}

In [ ]:
date_dim_nulls

{'date_key': 0,
 'year': 0,
 'month': 0,
 'week': 0,
 'day_of_week': 0,
 'is_weekend': 0,
 'is_holiday': 0,
 'holiday_name': 353}

In [ ]:
order_items.count()

203519

In [7]:
order_item_options.count()

193017

In [8]:
date_dim.count()

365

In [9]:
list(order_items.schema)

[StructField('APP_NAME', StringType(), True),
 StructField('RESTAURANT_ID', StringType(), True),
 StructField('CREATION_TIME_UTC', TimestampType(), True),
 StructField('ORDER_ID', StringType(), True),
 StructField('USER_ID', StringType(), True),
 StructField('PRINTED_CARD_NUMBER', LongType(), True),
 StructField('IS_LOYALTY', BooleanType(), True),
 StructField('CURRENCY', StringType(), True),
 StructField('LINEITEM_ID', StringType(), True),
 StructField('ITEM_CATEGORY', StringType(), True),
 StructField('ITEM_NAME', StringType(), True),
 StructField('ITEM_PRICE', DoubleType(), True),
 StructField('ITEM_QUANTITY', IntegerType(), True),
 StructField('ORDER_DATE', DateType(), True)]

In [10]:
list(order_item_options.schema)

[StructField('ORDER_ID', StringType(), True),
 StructField('LINEITEM_ID', StringType(), True),
 StructField('OPTION_GROUP_NAME', StringType(), True),
 StructField('OPTION_NAME', StringType(), True),
 StructField('OPTION_PRICE', DoubleType(), True),
 StructField('OPTION_QUANTITY', IntegerType(), True)]

In [11]:
list(date_dim.schema)

[StructField('date_key', StringType(), True),
 StructField('year', IntegerType(), True),
 StructField('month', IntegerType(), True),
 StructField('week', IntegerType(), True),
 StructField('day_of_week', StringType(), True),
 StructField('is_weekend', BooleanType(), True),
 StructField('is_holiday', BooleanType(), True),
 StructField('holiday_name', StringType(), True),
 StructField('DATE', DateType(), True)]

In [27]:
joined_df = order_items.join(order_item_options_grouped,on = (order_items['ORDER_ID'] == order_item_options_grouped['ORDER_ID'])
                             & (order_items['LINEITEM_ID'] == order_item_options_grouped['LINEITEM_ID']),
                             how ='left').drop(order_item_options_grouped['ORDER_ID']) \
                             .drop(order_item_options_grouped['LINEITEM_ID'])

joined_df = joined_df.withColumn("TOTAL_LINE_REVENUE",(F.col("ITEM_PRICE") * F.col("ITEM_QUANTITY")
                                                                              + F.coalesce(F.col("TOTAL_OPTIONS_PRICE"),F.lit(0))))

In [28]:
joined_df.filter((F.col("USER_ID") == '5ece77fe902ad501337b23fd') & (F.col("ORDER_ID") == '63d5d88aa26a29fa550df31f')) \
.orderBy("USER_ID").show(truncate=False)

+-------------+------------------------+-----------------------+------------------------+------------------------+-------------------+----------+--------+------------------------+-------------+-------------------+----------+-------------+----------+-------------------+-----------+------------------+
|APP_NAME     |RESTAURANT_ID           |CREATION_TIME_UTC      |ORDER_ID                |USER_ID                 |PRINTED_CARD_NUMBER|IS_LOYALTY|CURRENCY|LINEITEM_ID             |ITEM_CATEGORY|ITEM_NAME          |ITEM_PRICE|ITEM_QUANTITY|ORDER_DATE|TOTAL_OPTIONS_PRICE|NUM_OPTIONS|TOTAL_LINE_REVENUE|
+-------------+------------------------+-----------------------+------------------------+------------------------+-------------------+----------+--------+------------------------+-------------+-------------------+----------+-------------+----------+-------------------+-----------+------------------+
|Alltown Fresh|5fece84aadb0d51509c36f22|2023-01-28 20:23:06.225|63d5d88aa26a29fa550df31f|5ece77fe

In [29]:
date_dim.filter(F.col("DATE") == '2023-01-28').show()

+----------+----+-----+----+-----------+----------+----------+------------+----------+
|  date_key|year|month|week|day_of_week|is_weekend|is_holiday|holiday_name|      DATE|
+----------+----+-----+----+-----------+----------+----------+------------+----------+
|28-01-2023|2023|    1|   4|   Saturday|      true|     false|        NULL|2023-01-28|
+----------+----+-----+----+-----------+----------+----------+------------+----------+



In [30]:
joined_df_complete = joined_df.join(date_dim,on=joined_df['ORDER_DATE'] == date_dim['DATE'],how='inner') \
                              .drop("DATE")

In [32]:
# joined_df_complete.filter(F.col("USER_ID").isNotNull()) \
# .orderBy("USER_ID").show(truncate=False)

In [33]:
joined_df_complete.show(1)

+-------------+--------------------+--------------------+--------------------+--------------------+-------------------+----------+--------+--------------------+-------------+--------------------+----------+-------------+----------+-------------------+-----------+------------------+----------+----+-----+----+-----------+----------+----------+------------+
|     APP_NAME|       RESTAURANT_ID|   CREATION_TIME_UTC|            ORDER_ID|             USER_ID|PRINTED_CARD_NUMBER|IS_LOYALTY|CURRENCY|         LINEITEM_ID|ITEM_CATEGORY|           ITEM_NAME|ITEM_PRICE|ITEM_QUANTITY|ORDER_DATE|TOTAL_OPTIONS_PRICE|NUM_OPTIONS|TOTAL_LINE_REVENUE|  date_key|year|month|week|day_of_week|is_weekend|is_holiday|holiday_name|
+-------------+--------------------+--------------------+--------------------+--------------------+-------------------+----------+--------+--------------------+-------------+--------------------+----------+-------------+----------+-------------------+-----------+------------------+----

In [ ]:
#LTV

window_spec = Window.partitionBy("USER_ID").orderBy("ORDER_DATE")
# df_daily_order_total = df_filt_test.groupBy("USER_ID","ORDER_DATE").agg(F.sum("TOTAL_LINE_REVENUE").alias("DAILY_SPEND"))

df = joined_df_complete.withColumn("RUNNING_LTV",F.sum("TOTAL_LINE_REVENUE").over(window_spec))

In [42]:
df.select("USER_ID","ORDER_DATE","TOTAL_LINE_REVENUE","RUNNING_LTV") \
.filter(F.col("USER_ID") == '640f429571844bd533094761') \
.orderBy('ORDER_DATE').show(truncate=False)

+------------------------+----------+------------------+------------------+
|USER_ID                 |ORDER_DATE|TOTAL_LINE_REVENUE|RUNNING_LTV       |
+------------------------+----------+------------------+------------------+
|640f429571844bd533094761|2023-03-13|9.99              |9.99              |
|640f429571844bd533094761|2023-04-14|5.99              |22.97             |
|640f429571844bd533094761|2023-04-14|6.99              |22.97             |
|640f429571844bd533094761|2023-04-28|9.99              |32.96             |
|640f429571844bd533094761|2023-05-14|6.99              |47.940000000000005|
|640f429571844bd533094761|2023-05-14|7.99              |47.940000000000005|
|640f429571844bd533094761|2023-05-19|6.99              |60.92000000000001 |
|640f429571844bd533094761|2023-05-19|5.99              |60.92000000000001 |
|640f429571844bd533094761|2023-05-22|6.99              |67.91000000000001 |
|640f429571844bd533094761|2023-05-26|10.99             |89.89             |
|640f4295718

In [48]:
df_filt_test = joined_df_complete.select("USER_ID","ORDER_DATE","TOTAL_LINE_REVENUE") \
.filter(F.col("USER_ID") == '640f429571844bd533094761')


df_filt_test = df_filt_test.groupBy("USER_ID","ORDER_DATE").agg(F.sum("TOTAL_LINE_REVENUE").alias("DAILY_SPEND"))


df_filt_test = df_filt_test.withColumn("RUNNING_LTV",F.sum("DAILY_SPEND").over(window_spec))
df_filt_test.show()

+--------------------+----------+-----------+------------------+
|             USER_ID|ORDER_DATE|DAILY_SPEND|       RUNNING_LTV|
+--------------------+----------+-----------+------------------+
|640f429571844bd53...|2023-03-13|       9.99|              9.99|
|640f429571844bd53...|2023-04-14|      12.98|             22.97|
|640f429571844bd53...|2023-04-28|       9.99|             32.96|
|640f429571844bd53...|2023-05-14|      14.98|             47.94|
|640f429571844bd53...|2023-05-19|      12.98|             60.92|
|640f429571844bd53...|2023-05-22|       6.99|             67.91|
|640f429571844bd53...|2023-05-26|      21.98|             89.89|
|640f429571844bd53...|2023-06-13|       6.99|             96.88|
|640f429571844bd53...|2023-06-26|       6.99|103.86999999999999|
|640f429571844bd53...|2023-07-03|       6.99|110.85999999999999|
|640f429571844bd53...|2023-07-11|      16.98|127.83999999999999|
|640f429571844bd53...|2023-08-15|       6.99|134.82999999999998|
|640f429571844bd53...|202